# Phase 3A — Fine-tune ViT5-base on ViMedAQA

**Model:** `VietAI/vit5-base` (~270M params, Vietnamese Seq2Seq)  
**Task:** Abstractive Question Answering  
**Input:** `question: {q} context: {c}` → `{answer}`  

**Platform support:**
- ✅ **Kaggle** (P100 GPU, 12h session, 30h/week quota)
- ✅ **Google Colab** (T4 GPU, requires Drive mount)
- ✅ **Azure** (any GPU VM)

> ⚠️ **GPU REQUIRED.** Training on CPU is not feasible (~270M params).
> Estimated training time: **3-7 hours** (5 epochs on T4/P100).

## Cell 1 — Detect Platform & Verify GPU

In [ ]:
import os, sys, torch

# ── Detect platform ──
ON_KAGGLE = os.path.exists("/kaggle")
ON_COLAB  = "google.colab" in sys.modules or os.path.exists("/content")

if ON_KAGGLE:
    PLATFORM = "Kaggle"
    # Tự động tìm thư mục chứa train.json trong /kaggle/input
    base_input = "/kaggle/input"
    subdirs = [os.path.join(base_input, d) for d in os.listdir(base_input) if os.path.isdir(os.path.join(base_input, d))]
    
    # Ưu tiên thư mục có chứa train.json, nếu không có thì mặc định /kaggle/input
    DATA_ROOT = next((d for d in subdirs if os.path.isfile(os.path.join(d, "train.json"))), base_input)
    OUTPUT_ROOT = "/kaggle/working"
elif ON_COLAB:
    PLATFORM = "Colab"
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_ROOT = "/content/drive/MyDrive/vimedaq-project/data/processed"
    OUTPUT_ROOT = "/content/drive/MyDrive/vimedaq-project"
else:
    PLATFORM = "Local"
    DATA_ROOT = os.path.join("..", "data", "processed")
    OUTPUT_ROOT = os.path.join("..")  # project root

print(f"Platform : {PLATFORM}")
print(f"Data root: {DATA_ROOT}")
print(f"Output   : {OUTPUT_ROOT}")

# ── Verify GPU ──
if not torch.cuda.is_available():
    raise RuntimeError(
        "⚠️ NO GPU DETECTED!\n"
        "  Kaggle : Settings → Accelerator → GPU P100\n"
        "  Colab  : Runtime → Change runtime type → T4 GPU"
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"\n✅ GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)")

# ── Create output dirs ──
CHECKPOINT_DIR = os.path.join(OUTPUT_ROOT, "checkpoints", "vit5")
BEST_MODEL_DIR = os.path.join(CHECKPOINT_DIR, "best")
LOGS_DIR       = os.path.join(OUTPUT_ROOT, "logs")
os.makedirs(BEST_MODEL_DIR, exist_ok=True)
os.makedirs(LOGS_DIR, exist_ok=True)
print(f"Checkpoints: {BEST_MODEL_DIR}")
print(f"Logs       : {LOGS_DIR}")

## Cell 2 — Install Dependencies

In [ ]:
# ===== CELL 2 — Install Dependencies =====
!pip install transformers==4.46.3 sentencepiece==0.2.0 datasets evaluate rouge-score accelerate -q

import transformers
print(f"✅ transformers {transformers.__version__}")


## Cell 3 — Load Data

Load the train/val splits produced by Phase 1 (`01_data_exploration.ipynb`).  

**Kaggle users:** Upload `train.json` and `val.json` as a Kaggle Dataset,  
then set the correct path in `DATA_ROOT` above (e.g. `/kaggle/input/your-dataset-name/`).

In [ ]:
import json
import os
from datasets import Dataset

# --- 1. Tự động quét toàn bộ hệ thống để tìm file ---
train_path = None

# Quét trên Kaggle trước
if os.path.exists("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        for file in files:
            if file == "train.json":
                train_path = os.path.join(root, file)
                break
        if train_path: break

# 2. Nếu không có ở Kaggle thì dự phòng Colab/Local
if not train_path:
    fallback_paths = [
        "/content/drive/MyDrive/vimedaq-project/data/processed/train.json",
        "../data/processed/train.json"
    ]
    for path in fallback_paths:
        if os.path.isfile(path):
            train_path = path
            break

if not train_path:
    raise FileNotFoundError("Đã lật tung máy lên tìm nhưng vẫn không thấy train.json!")

val_path = train_path.replace("train.json", "val.json")

def load_split(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

# --- 3. Load data ---
train_data = load_split(train_path)
val_data   = load_split(val_path)

train_dataset = Dataset.from_list(train_data)
val_dataset   = Dataset.from_list(val_data)

print(f"✅ Bắt được data tại: {train_path}")
print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")
print(f"Columns: {train_dataset.column_names}")

## Cell 4 — Tokenizer & Preprocessing

In [ ]:
# ===== CELL 4 — Tokenizer & Preprocessing =====
from transformers import T5Tokenizer

MODEL_NAME = "VietAI/vit5-base"
MAX_INPUT_LEN = 512
MAX_TARGET_LEN = 128

# Load tokenizer
tokenizer = T5Tokenizer.from_pretrained(MODEL_NAME)

# Verify tokenizer hoạt động với tiếng Việt
test = "Xin chào Việt Nam"
ids = tokenizer.encode(test)
print(f"IDs: {ids}")
print(f"Decoded: '{tokenizer.decode(ids)}'")
print(f"Vocab size: {tokenizer.vocab_size}")

def preprocess(examples):
    inputs = [
        f"question: {q} context: {c}"
        for q, c in zip(examples["question"], examples["context"])
    ]
    targets = examples["answer"]

    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LEN,
        padding="max_length",
        truncation=True,
    )
    labels = tokenizer(
        text_target=targets,
        max_length=MAX_TARGET_LEN,
        padding="max_length",
        truncation=True,
    )
    labels["input_ids"] = [
        [(tok if tok != tokenizer.pad_token_id else -100) for tok in label]
        for label in labels["input_ids"]
    ]
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Tokenizing train set...")
train_tokenized = train_dataset.map(
    preprocess, batched=True, remove_columns=train_dataset.column_names,
)
print("Tokenizing val set...")
val_tokenized = val_dataset.map(
    preprocess, batched=True, remove_columns=val_dataset.column_names,
)
print(f"✅ Done! Train: {train_tokenized.shape} | Val: {val_tokenized.shape}")


## Cell 5 — Load Model

In [ ]:
from transformers import AutoModelForSeq2SeqLM
SAVED_MODEL_PATH = "/kaggle/input/notebooks/aeaefff123/03a-train-vit5/checkpoints/vit5/best"
model = AutoModelForSeq2SeqLM.from_pretrained(SAVED_MODEL_PATH)
model = model.to("cuda")

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params    : {total_params / 1e6:.1f}M")
print(f"Trainable params: {trainable_params / 1e6:.1f}M")
print(f"VRAM usage      : {torch.cuda.memory_allocated() / 1e9:.2f} GB")
model.config.decoder_start_token_id = tokenizer.pad_token_id
model.generation_config.decoder_start_token_id = tokenizer.pad_token_id

## Cell 6 — Training Configuration

OOM-safe config for T4/P100 (16 GB VRAM).  
If OOM occurs: reduce `BATCH_SIZE` to 2, increase `GRAD_ACCUM` to 8.

In [ ]:
# ===== CELL 6 — Training Configuration =====
from transformers import (
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq,
)
import evaluate
import numpy as np

BATCH_SIZE    = 4
GRAD_ACCUM    = 4
LEARNING_RATE = 3e-4
NUM_EPOCHS    = 5
WARMUP_STEPS  = 100
LOCAL_CKPT_DIR = os.path.join(OUTPUT_ROOT, "vit5_local_ckpts")
training_args = Seq2SeqTrainingArguments(
    output_dir=LOCAL_CKPT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    eval_strategy="epoch",
    save_strategy="epoch",          # Phải match với eval_strategy
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    greater_is_better=True,
    predict_with_generate=True,
    generation_max_length=MAX_TARGET_LEN,
    fp16=True,
    logging_dir=LOGS_DIR,
    logging_steps=50,
    report_to="none",
    dataloader_num_workers=2,
)
print("✅ Training args configured.")
print(f"   Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")
print(f"   Epochs: {NUM_EPOCHS}")
print(f"   Eval: mỗi epoch (5 lần total)")


## Cell 7 — Compute Metrics (ROUGE during training)

In [ ]:
rouge_metric = evaluate.load("rouge")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    # Fix: clip token IDs ngoài phạm vi vocab
    preds = np.where(
        (preds >= 0) & (preds < tokenizer.vocab_size),
        preds, tokenizer.pad_token_id
    )
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    decoded_preds  = [p.strip() for p in decoded_preds]
    decoded_labels = [l.strip() for l in decoded_labels]
    result = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {k: round(v, 4) for k, v in result.items()}

print("✅ compute_metrics defined.")


## Cell 8 — Train

⏱️ **Estimated time:** 3-7 hours (5 epochs on T4/P100).  
⚠️ Keep browser tab **active**. If session disconnects, re-run Cells 1-7  
then use `trainer.train(resume_from_checkpoint=...)` — see Cell 8b below.

In [5]:
from transformers.trainer_utils import get_last_checkpoint

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

#last_checkpoint = get_last_checkpoint(LOCAL_CKPT_DIR)

#Do lần train trước đã thành công nhưng bị lỗi ở các cell cuối dẫn đến chương trình bị crash, nên giờ làm lại để sạch file
last_checkpoint = "/kaggle/input/datasets/aeaefff123/trainingresult/vit5_local_ckpts/checkpoint-2945"
if last_checkpoint:
    print(f"🚀 Resume từ: {last_checkpoint}")
else:
    print("🚀 Train từ đầu")

trainer.train(resume_from_checkpoint=last_checkpoint)

print("\n" + "=" * 60)
print("✅ Training complete!")


NameError: name 'DataCollatorForSeq2Seq' is not defined

## Cell 8b — Resume Training (only if disconnected)

Skip this cell if training completed normally.  
If session was disconnected, re-run Cells 1-7, then run this cell.

In [ ]:
# Uncomment and set the correct checkpoint path:
# import glob
# ckpts = sorted(glob.glob(os.path.join(LOCAL_CKPT_DIR, "checkpoint-*")))
# print("Available checkpoints:", ckpts)
# trainer.train(resume_from_checkpoint=ckpts[-1])  # resume from latest

## Cell 9 — Save Best Model (CRITICAL)

Save the best model and training log to persistent storage.

In [ ]:
import pandas as pd

# Save best model
trainer.save_model(BEST_MODEL_DIR)
tokenizer.save_pretrained(BEST_MODEL_DIR)
print(f"✅ Best model saved: {BEST_MODEL_DIR}")

# List saved files
for f in os.listdir(BEST_MODEL_DIR):
    size = os.path.getsize(os.path.join(BEST_MODEL_DIR, f))
    print(f"   {f} ({size/1e6:.1f} MB)")

# Save training log
log_path = os.path.join(LOGS_DIR, "vit5_training_log.csv")
pd.DataFrame(trainer.state.log_history).to_csv(log_path, index=False)
print(f"\n✅ Training log saved: {log_path}")

# Print final metrics
print("\n── Final Eval Metrics ──")
for entry in reversed(trainer.state.log_history):
    if "eval_rougeL" in entry:
        print(f"  eval_rouge1 : {entry.get('eval_rouge1', 'N/A')}")
        print(f"  eval_rouge2 : {entry.get('eval_rouge2', 'N/A')}")
        print(f"  eval_rougeL : {entry.get('eval_rougeL', 'N/A')}")
        break

## Cell 10 — Sanity Check Inference

Quick test: generate answers for a few validation samples.

In [ ]:
model.eval()
samples = val_data[:5]

print("=" * 70)
print("SANITY CHECK — ViT5 Fine-tuned Inference")
print("=" * 70)

for idx, item in enumerate(samples):
    input_text = f"question: {item['question']} context: {item['context']}"
    inputs = tokenizer(
        input_text, return_tensors="pt",
        max_length=MAX_INPUT_LEN, truncation=True,
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=MAX_TARGET_LEN,
            num_beams=4,
            early_stopping=True,
            decoder_start_token_id=0
        )

    pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"\n[{idx+1}] Q: {item['question'][:80]}")
    print(f"    Pred: {pred[:120]}")
    print(f"    Ref : {item['answer'][:120]}")
    print("-" * 70)

print("\n✅ Sanity check complete.")
print("If predictions are valid Vietnamese text, Phase 3A is DONE!")

## Cell 11 — Download Model (Kaggle only)

On Kaggle, outputs are saved to `/kaggle/working/`.  
After the notebook finishes, go to **Output** tab → **Download All**.

In [ ]:
if ON_KAGGLE:
    import shutil
    # Compress model for easier download
    archive_path = os.path.join(OUTPUT_ROOT, "vit5_best_model")
    shutil.make_archive(archive_path, "zip", BEST_MODEL_DIR)
    print(f"✅ Model archived: {archive_path}.zip")
    size_mb = os.path.getsize(f"{archive_path}.zip") / 1e6
    print(f"   Size: {size_mb:.0f} MB")
    print("   Go to Output tab → Download All")
else:
    print(f"Model already saved at: {BEST_MODEL_DIR}")
    print("No additional download needed.")